# Zadanie: Random Forest
Podziel zbiór na treningowy i testowy, pamiętaj o stratyfikacji. Następnie naucz model Random Forest, z którego wyciągnij feature importance. Na podstawie tego wykonaj selekcję cech i weź jedynie te, których ważność jest większa niż 0.001. Nauczy nowy model Random Forest z wyborem hiperperparametrów, korzystając z GridSearch. Wykorzystaj poznane techniki do wektoryzacji.

Importy bibliotek

In [ ]:
import numpy as np
import pandas as pd
import string
import nltk
import itertools
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

Import danych.

In [ ]:
spam_dataset = pd.read_csv('spam.csv', encoding = "ISO-8859-1", usecols=[0, 1], names=['Spam', 'Text'],skiprows=1)
spam_dataset['Spam'] = spam_dataset['Spam'].replace(['ham', 'spam'], [0, 1])

Funkcja przetwarzajaca tekst.

In [ ]:
stopwords = nltk.corpus.stopwords.words("english")
lemmater = nltk.WordNetLemmatizer()

def przygotowanie_tekstu(text):
    cleaned = ''.join([char for char in text if char not in string.punctuation])
    clean_text = cleaned.lower()
    tokens = nltk.word_tokenize(clean_text)
    without_stop = [word for word in tokens if word not in stopwords]
    lemmatized = [lemmater.lemmatize(word) for word in without_stop]
    
    return lemmatized

podział na zestaw testowy i treningowy

In [ ]:
spam_dataset['Lemmatized_Text'] = spam_dataset['Text'].apply(przygotowanie_tekstu)

X = spam_dataset['Lemmatized_Text'].apply(lambda x: ' '.join(x))
y = spam_dataset['Spam'].astype('int')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

Pierwsza wektoryzacja dla wszsytkich danych.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

Pierwszy podstawowy model uczącu sie na wszsytkich danych.

In [ ]:
RandomForest_1 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
RandomForest_1.fit(X_train_tfidf, y_train)

Wyciągnięcie feature importance oraz odfiltrowanie tylko tych tym o importance > od 0.001 

In [ ]:
feature_names = tfidf.get_feature_names_out()
importances = RandomForest_1.feature_importances_
feature_importance = pd.DataFrame({'feature': feature_names, 'importance': importances})
selected_features_df = feature_importance[feature_importance['importance'] > 0.001]
selected_words = selected_features_df['feature'].tolist()

In [ ]:
print(f"Liczba cech przed selekcją: {len(feature_names)}")
print(f"Liczba cech po selekcji (>0.001): {len(selected_words)}")

Liczba słów zmniejszyła się o ponad 96%, to oznacza że raptem niewiele ponad 3% słów ma znaczenie, reszta to szum.

Druga wektoryzacja tylko na wybranych, ważnych słowach

In [ ]:
selected_tfidf = TfidfVectorizer(vocabulary=selected_words)

X_train_selected = selected_tfidf.fit_transform(X_train)
X_test_selected = selected_tfidf.transform(X_test)

In [ ]:
RandomForest_2 = RandomForestClassifier(random_state=42, n_jobs=-1)

In [ ]:
param_grid = {
    'n_estimators':[100, 200],
    'max_depth': [None, 15, 20],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf':[1,2,4],
    'max_features': ['sqrt', 'log2']
}

In [ ]:
grid_RF = GridSearchCV(RandomForest_2, param_grid, cv=3, n_jobs=-1, scoring='accuracy', verbose=1)
grid_RF.fit(X_train_selected, y_train)

In [ ]:
print("Najlepsze parametry znalezione przez GridSearch:")
print(grid_RF.best_params_)

print(f"\nWynik (Accuracy): dla podstawowego modelu. {RandomForest_1.score(X_test_tfidf, y_test):.4f}")
print(f"\nWynik (Accuracy) dla modelu po selekcji cech i GridSearch: {grid_RF.score(X_test_selected, y_test):.4f}")

Wydawać by się mogło że wzrost z 0.9767 w modelu podstawowym do 0.9776 po selekcji i gridSearch to mało, jednak należy pamiętać że dane zostały odchudzone o ponad 96%, więc model uczy się zdecydowanie szybciej a mimo to wynik jest nieznacznie lepszy